# **Transformers and XAI: practice**
**Welcome to the final practice of the course, everyone!**

Before we start, I want to say thank you! This is my first publicly available course, and it was important to me to make it interesting. Your feedback, remarks and ideas made it better and keep making it better, so feel free to share them in the course chats. I am grateful to you for your attention and your persistence with the tasks. You are the best!

**Let us move on to the lesson! :)**

Interpreting models is an interesting and exciting task, and transformers with their attention mechanism are no exception. However, when working with them, what matters is the **quality of pre-training**.

As a rule, if you rely on attention maps when analysing importance in a model, you need to use a *well pre-trained* model for your task. This will guarantee you that the attention of the model is well tuned to your data and, as a consequence, that you really *can try* to extract working hypotheses about the data from it.  

However, attention is not the only thing there is! Methods such as SHAP, LIME, Integrated Gradients and others can be useful too.

**In this lesson we will:**

- Learn to visualize the attention mechanism (attention weights);
- Apply interpretation methods (such as SHAP) to analyse the decisions made by the model;
- Get to know the tooling that lets you visualize transformer-type models in a couple of lines of code;
- Discuss the importance of interpreting models in real applications and look at practical cases.

**It will be beautiful!** Happy coding! :)

![](https://ucarecdn.com/1a50e7f8-b09b-4b85-b928-b50efe5ecbba/)

Let us start by installing all the libraries we need. Do not be alarmed, this may take 2-5 minutes.

In [ ]:
!pip install transformers shap transformers-interpret ferret-xai -q # As always, let us pull in all the libraries we need

And let us import the entities we need.

In [ ]:
# MODEL
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import torch

# XAI Libs
from transformers_interpret import SequenceClassificationExplainer
from ferret import Benchmark
import shap

# VIZ
import numpy as np
import matplotlib.pyplot as plt

We will work on a classical and very beautiful task — the task of text sentiment classification. To save time, we will take the pre-trained `distilbert` model.

**DistilBERT** is a smaller and more efficient model than the base BERT (Bidirectional Encoder Representations from Transformers). It has 40% fewer parameters than google-bert/bert-base-uncased and runs 60% faster, while retaining more than 95% of the performance of the original model.

In [ ]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(model_name, attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Let us define the text to work with. You can also set your own, but to complete the tasks on Stepik, use the original one.

In [ ]:
data = ["You look terrible! Maybe you should rest?"]

In [ ]:
txt_tokens = tokenizer(data[0], return_tensors="pt", add_special_tokens=True) # Tokenize the words
labels = tokenizer.tokenize(data[0], add_special_tokens=True) # Save the labels for building the future attention map

**Which token id corresponds to the first word?**
Your answer here

We will build the attention by taking the median over the transformer outputs. By default, the number of attention maps in the model is **num_layers*num_heads**.

**Check how many attention maps the distillbert model has?**

In [ ]:
# Your answer here

Since the inference of a transformer is a demanding task, to obtain the attention maps you need to specify a special hyperparameter — `output_attentions=True`. For the visualization we use the median value over all the heads.

**Note:** No one stops you from taking the mean, the minimum, the maximum or another statistic. It is hard to say which one will be the most informative, however the median is the one most often used by default in papers.

In [ ]:
predictions = model(**txt_tokens, output_attentions=True)

In [ ]:
all_attentions = np.array([i.detach().numpy() for i in predictions.attentions[:]]).squeeze(1)

med_layers_att = np.median(all_attentions, axis=0)
med_heads_att = np.median(med_layers_att, axis=0)


In [ ]:
# Let us visualize the attention

fig, ax = plt.subplots(figsize=(6, 12))
ax.imshow(np.median(all_attentions[3], axis=0))


ax.set_xticks(np.arange(len(labels)), labels=labels)
ax.set_yticks(np.arange(len(labels)), labels=labels);

To smooth the estimate you can use one more averaging, but even at this stage it is clear that the median map is not entirely informative. This happens because every head can take on different patterns of the input sequence. Besides that, the model was trained to work with the sentiment of reviews, and not with sentiment in general.

But the downsides do not mean that we cannot continue the investigation. Other methods can also be applied to a transformer-type model.


Let us start with shap.

### **shap X transformers**

In [ ]:
classifier = pipeline('text-classification', top_k=None, model=model, tokenizer=tokenizer)
explainer = shap.Explainer(classifier)

print('Predictions')
print(classifier(data))

# Let us compute the shap values
shap_values = explainer(data)

In [ ]:
shap.plots.text(shap_values)

**Analyse the shap plot the way we did before. What is the prediction of the model?**

**What is the base value of the NEGATIVE class?**

Let us look at one more type of plot — the barplot. You can visualize the influence of the tokens on the "POSITIVE" and "NEGATIVE" classes separately.

In [ ]:
# The influence of the tokens on the Negative class
shap.plots.bar(shap_values[0, :, "NEGATIVE"], max_display=10)

In [ ]:
# The influence of the tokens on the Positive class
shap.plots.bar(shap_values[0, :, "POSITIVE"], max_display=10)

### **transformers-interpret X transformers**

One more mechanical implementation of a way to look inside the model is the`transformers-interpret` library. As you can tell from the name, it was developed specifically for transformer-type models.

The upsides:
- Transformers Interpret lets you explain any transformer model in just two lines
- Explanations are available both for text models and for CV computer vision models
- The visualizations are available as easily saved png and html files

The downsides:
- The wrapper gives you less control over how the explanation is obtained

Overall, this library is an excellent way to get an explanation quickly. It saves time and, on top of that, makes it easier to build a report on the explanations thanks to the convenient file saving.

For examples with MultiLabel, Zero Shot Classification, Question Answering, Token Classification (NER) and Image Classification, go [here](https://github.com/cdpierse/transformers-interpret/tree/master).

In [ ]:
cls_explainer = SequenceClassificationExplainer(model, tokenizer) # following the philosophy of the library, we declare the classifier
word_attributions = cls_explainer(data[0]) # we get the attributions in a couple of lines

In [ ]:
cls_explainer.visualize("distilbert_viz.html", true_class='NEGATIVE') # we carry out the visualization of the attributions

In [ ]:
fig, ax = plt.subplots()

y_pos = np.arange(len(word_attributions))
word_attributions_numbers = [i[1] for i in word_attributions]
word_attributions_labels = [i[0] for i in word_attributions]

ax.barh(y_pos, word_attributions_numbers)
ax.set_yticks(y_pos, labels=word_attributions_labels)
ax.set_title('Word attributions with transformers-interpret')
plt.show()

### **FERRET x transformers**

As you can notice, the attributions obtained by different methods within one and the same model are different. We have seen this throughout the course and we are seeing it now. It is impossible to say which one is better. But perhaps everything is still ahead of us =)


The upsides of the framework:
- fast comparative analysis of the attributions of different explanation methods
- support for audio data and explanations for it

The downsides:
- Just like with transformers-interpet, the wrapper gives you less control over how the explanation is obtained

In [ ]:
bench = Benchmark(model, tokenizer)

explanations = bench.explain('You look terrible! Maybe you should rest?', target=0)
evaluations = bench.evaluate_explanations(explanations, target=0)

In [ ]:
bench.show_table(explanations)

**Which token has the highest score in the ferret table for its influence on the prediction?**

 A few more libraries for working with transformers are waiting for you on the platform. 🏎

 Thank you for your work, everyone! I am looking forward to seeing you in new courses! :)